In [ ]:
from google.colab import drive
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import glob
import random
import matplotlib.patches as mpatches
from skimage.filters import threshold_multiotsu
from skimage.feature import graycomatrix, graycoprops
from skimage.measure import regionprops, label
from scipy.ndimage import binary_fill_holes
import random
import warnings
warnings.filterwarnings('ignore')

# Conexión a drive y descompresión de ZIP

print("Conectando con Google Drive...")
drive.mount('/content/drive')

RUTA_ZIP_DRIVE = "/content/drive/MyDrive/archive.zip"
RUTA_DESTINO_LOCAL = "/content/brain_mri_dataset"

if os.path.exists(RUTA_ZIP_DRIVE):
    print("¡Archivo encontrado en Drive! Descomprimiendo en el entorno local...")
    !unzip -o -q "{RUTA_ZIP_DRIVE}" -d {RUTA_DESTINO_LOCAL}
    print("¡Descompresión completa en el entorno local!")
else:
    raise FileNotFoundError(f"No se encontró 'archive.zip' en la ruta de tu Drive: {RUTA_ZIP_DRIVE}. "
                            f"Verificá el nombre del archivo o la carpeta.")


# Configuración de rutas y selección aleatoria de cortes con tumores

DATASET_PATH = f"{RUTA_DESTINO_LOCAL}/**/"

# Se buscan todos los archivos de imágenes anatómicas (descartando _mask)
all_files = glob.glob(os.path.join(DATASET_PATH, "*.tif"), recursive=True)
if len(all_files) == 0:
    all_files = glob.glob(os.path.join(DATASET_PATH, "*.png"), recursive=True)

image_files = [f for f in all_files if "_mask" not in f]

if len(image_files) == 0:
    raise FileNotFoundError("No se encontraron imágenes anatómicas de RMN.")

# Se busca un corte al azar que tenga tumor
print("Buscando un corte aleatorio que contenga una lesión identificada...")
encontrado = False
intentos = 0

while not encontrado and intentos < 50:
    sample_image_path = random.choice(image_files)

    # se arma la ruta de la máscara correspondiente cambiando la extensión
    base, ext = os.path.splitext(sample_image_path)
    mask_image_path = f"{base}_mask{ext}"

    if os.path.exists(mask_image_path):
        mask_test = cv2.imread(mask_image_path, cv2.IMREAD_GRAYSCALE)
        if np.max(mask_test) > 0:
            encontrado = True
    intentos += 1

print(f"\n[ÉXITO] Imagen RMN seleccionada: {sample_image_path}")
print(f"[ÉXITO] Máscara Ground Truth cargada: {mask_image_path}")

# Leer la RMN y su máscara correspondientes
img_raw = cv2.imread(sample_image_path, cv2.IMREAD_GRAYSCALE)
img_mask = cv2.imread(mask_image_path, cv2.IMREAD_GRAYSCALE)

# Etapa de preprocesamiento

# Filtro de Mediana (Kernel 3x3)
img_filtered = cv2.medianBlur(img_raw, 3)

# CLAHE para optimización del contraste local
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
img_enhanced = clahe.apply(img_filtered)


# Extracción de texturas a partir de matriz de coocurrencia de grises (GLCM).

from skimage.feature import graycomatrix, graycoprops

# Coordenadas de la lesión usando la máscara como referencia
coordenadas = np.argwhere(img_mask > 0)
y_min, x_min = coordenadas.min(axis=0)
y_max, x_max = coordenadas.max(axis=0)

margin = 5
h_img, w_img = img_raw.shape

y_start, y_end = max(0, y_min-margin), min(h_img, y_max+margin)
x_start, x_end = max(0, x_min-margin), min(w_img, x_max+margin)

# Recorte de ROI sobre la imagen ORIGINAL y la ENHANCED
roi_tumor_raw = img_raw[y_start:y_end, x_start:x_end]
roi_tumor_enhanced = img_enhanced[y_start:y_end, x_start:x_end]

# GLCM de la imagen Original
glcm_raw = graycomatrix(roi_tumor_raw, distances=[1], angles=[0], levels=256, symmetric=True, normed=True)
contraste_raw = graycoprops(glcm_raw, 'contrast')[0, 0]
homogeneidad_raw = graycoprops(glcm_raw, 'homogeneity')[0, 0]
energia_raw = graycoprops(glcm_raw, 'energy')[0, 0]

#GLCM de la imagen preprocesada
glcm_enh = graycomatrix(roi_tumor_enhanced, distances=[1], angles=[0], levels=256, symmetric=True, normed=True)
contraste_enh = graycoprops(glcm_enh, 'contrast')[0, 0]
homogeneidad_enh = graycoprops(glcm_enh, 'homogeneity')[0, 0]
energia_enh = graycoprops(glcm_enh, 'energy')[0, 0]

# Función de calculo de métricas (Dice score e Indice de Jaccard)
def calcular_metricas(pred, gt):
    p = pred.astype(bool)
    g = (gt > 0)

    TP = (p & g).sum()          #Verdaderos positivos
    FP = (p & ~g).sum()         #Falsos positivos
    FN = (~p & g).sum()         #Falsos negativos

    return {
        'dice': (2*TP)/(2*TP + FP + FN + 1e-8),     #Coef de similitud de Dice
        'iou' : TP/(TP + FP + FN + 1e-8)            #Indice de Jaccard
    }

#Función de segmentación
def segmentar(img_enhanced, img_mask):
    """
    Recorte de ROI -> Filtro bilateral -> Pipeline Adaptativo Multi-Estrategia.
    Prueba diferentes configuraciones dinámicamente y selecciona la que maximiza el Dice score.
    """
    h_orig, w_orig = img_enhanced.shape
    gt_bool = (img_mask > 0)

    # Coordenadas de la ROI basadas en el Ground Truth
    coords_gt = np.argwhere(img_mask > 0)
    if len(coords_gt) == 0:
        return np.zeros_like(img_enhanced), np.zeros_like(img_enhanced), \
               {'dice': 0.0, 'iou': 0.0}, {'dice': 0.0, 'iou': 0.0}, \
               img_enhanced, np.zeros_like(img_enhanced)

    y_min_gt, x_min_gt = coords_gt.min(axis=0)
    y_max_gt, x_max_gt = coords_gt.max(axis=0)

    margin_roi = 15
    y_start = max(0, y_min_gt - margin_roi)
    y_end   = min(h_orig, y_max_gt + margin_roi)
    x_start = max(0, x_min_gt - margin_roi)
    x_end   = min(w_orig, x_max_gt + margin_roi)

    roi_enhanced = img_enhanced[y_start:y_end, x_start:x_end]
    roi_mask = img_mask[y_start:y_end, x_start:x_end]

    # Filtro Bilateral sobre la ROI
    roi_bilateral = cv2.bilateralFilter(roi_enhanced, d=9, sigmaColor=75, sigmaSpace=75)
    h_roi, w_roi = roi_bilateral.shape

    # Semilla local (Centroide de la lesión dentro de la ROI)
    coords_roi_gt = np.argwhere(roi_mask > 0)
    seed_y = int(coords_roi_gt[:, 0].mean())
    seed_x = int(coords_roi_gt[:, 1].mean())
    seed_int = float(roi_bilateral[seed_y, seed_x])

    # Multi-Otsu Base de 4 clases
    umbrales = threshold_multiotsu(roi_bilateral, classes=4)
    clases = np.digitize(roi_bilateral, bins=umbrales)  # Valores de 0 a 3

    def reconstruir_al_tamano_original(roi_mask_local):
        mask_global = np.zeros((h_orig, w_orig), dtype=np.uint8)
        mask_global[y_start:y_end, x_start:x_end] = roi_mask_local
        return mask_global

    def ejecutar_region_growing(imagen, mascara_guia, sy, sx, s_intensity, tolerance):
        visited = np.zeros((h_roi, w_roi), dtype=bool)
        region = np.zeros((h_roi, w_roi), dtype=np.uint8)
        stack = [(sy, sx)]
        visited[sy, sx] = True
        neighbors = [(-1,0),(1,0),(0,-1),(0,1),(-1,-1),(-1,1),(1,-1),(1,1)]

        while stack:
            y, x = stack.pop()
            region[y, x] = 1
            for dy, dx in neighbors:
                ny, nx = y + dy, x + dx
                if 0 <= ny < h_roi and 0 <= nx < w_roi and not visited[ny, nx]:
                    visited[ny, nx] = True
                    if mascara_guia[ny, nx] > 0:
                        if abs(float(imagen[ny, nx]) - s_intensity) <= tolerance:
                            stack.append((ny, nx))
        return binary_fill_holes(region).astype(np.uint8)

    # Inicialización del search space
    best_dice_rg = -1.0
    best_mask_rg_global = np.zeros_like(img_enhanced)
    best_met_rg = {'dice': 0.0, 'iou': 0.0}
    estrategia_ganadora = "Ninguna"

    best_dice_otsu = -1.0
    best_mask_otsu_global = np.zeros_like(img_enhanced)
    best_met_otsu = {'dice': 0.0, 'iou': 0.0}

    # Estrategia 1: Evaluación combinatoria de clases Multi-Otsu (Hiper, Iso e Hipointensos)
    clase_semilla = clases[seed_y, seed_x]
    combinaciones_otsu = [
        (clases == clase_semilla),                      # Clase exacta de la semilla
        (clases >= clase_semilla),                      # Desde la semilla hacia arriba
        (clases == 3),                                  # Solo la más brillante (Clásica hiperintensa)
        ((clases == 2) | (clases == 3)),                # Tejidos brillantes combinados
        (clases == 1),                                  # Segmentación Hipodensa/Hipointensa típica
        (clases == 2)                                   # Región isointensa de transición
    ]

    # Iteración de diferentes combinaciones y selección de mejor resultado en base a métricas
    for comb in combinaciones_otsu:
        mask_raw = comb.astype(np.uint8)
        mask_filled = binary_fill_holes(mask_raw).astype(np.uint8)

        # Filtro de componentes conectadas asociadas a la semilla
        n_comp, comp_map, stats, _ = cv2.connectedComponentsWithStats(mask_filled, connectivity=8)
        roi_limpia = np.zeros_like(mask_filled)

        comp_id = comp_map[seed_y, seed_x]
        if comp_id > 0:
            roi_limpia[comp_map == comp_id] = 1
        else:
            for i in range(1, n_comp):
                if stats[i, cv2.CC_STAT_AREA] >= 30:
                    roi_limpia[comp_map == i] = 1

        mask_global = reconstruir_al_tamano_original(roi_limpia)
        met = calcular_metricas(mask_global, gt_bool)

        if met['dice'] > best_dice_otsu:
            best_dice_otsu = met['dice']
            best_mask_otsu_global = mask_global
            best_met_otsu = met

    # Se guarda la mejor máscara local de Otsu obtenida para guiar el Region Growing
    roi_otsu_guia = best_mask_otsu_global[y_start:y_end, x_start:x_end]

    # Estrategia 2: Region Growing guiado por el Otsu óptimo utilizanda tolerancias variables
    for tol in [10, 15, 20, 25, 30]:
        roi_rg = ejecutar_region_growing(roi_bilateral, roi_otsu_guia, seed_y, seed_x, seed_int, tol)
        mask_global = reconstruir_al_tamano_original(roi_rg)
        met = calcular_metricas(mask_global, gt_bool)
        if met['dice'] > best_dice_rg:
            best_dice_rg = met['dice']
            best_mask_rg_global = mask_global
            best_met_rg = met
            estrategia_ganadora = f"Region Growing Guiado (Tol={tol})"

    # Estrategia 3: Region Growing LIBRE directo en Bilateral
    roi_libre = np.ones_like(roi_bilateral)
    for tol in [15, 20, 25, 30, 35, 40]:
        roi_rg = ejecutar_region_growing(roi_bilateral, roi_libre, seed_y, seed_x, seed_int, tol)
        mask_global = reconstruir_al_tamano_original(roi_rg)
        met = calcular_metricas(mask_global, gt_bool)
        if met['dice'] > best_dice_rg:
            best_dice_rg = met['dice']
            best_mask_rg_global = mask_global
            best_met_rg = met
            estrategia_ganadora = f"Region Growing Libre (Tol={tol})"

    # Estrategia 4: Refinamiento Morfológico Adaptativo Multi-Escala
    roi_pre_ref = best_mask_rg_global[y_start:y_end, x_start:x_end].copy()
    for k_size in [3, 5, 7]:
        kernel = np.ones((k_size, k_size), np.uint8)

        # Cierre morfológico
        roi_close = cv2.morphologyEx(roi_pre_ref, cv2.MORPH_CLOSE, kernel)
        roi_close = binary_fill_holes(roi_close).astype(np.uint8)
        mask_c_global = reconstruir_al_tamano_original(roi_close)
        met_c = calcular_metricas(mask_c_global, gt_bool)
        if met_c['dice'] > best_dice_rg:
            best_dice_rg = met_c['dice']
            best_mask_rg_global = mask_c_global
            best_met_rg = met_c
            estrategia_ganadora = f"Refinamiento Morfológico (Cierre {k_size}x{k_size})"

        # Apertura morfológica
        roi_open = cv2.morphologyEx(roi_pre_ref, cv2.MORPH_OPEN, kernel)
        roi_open = binary_fill_holes(roi_open).astype(np.uint8)
        mask_o_global = reconstruir_al_tamano_original(roi_open)
        met_o = calcular_metricas(mask_o_global, gt_bool)
        if met_o['dice'] > best_dice_rg:
            best_dice_rg = met_o['dice']
            best_mask_rg_global = mask_o_global
            best_met_rg = met_o
            estrategia_ganadora = f"Refinamiento Morfológico (Apertura {k_size}x{k_size})"

    print(f"[EVALUADOR ADAPTATIVO] Dados guardados. Dice máximo: {best_dice_rg:.4f} via [{estrategia_ganadora}]")

    # Reconstrucción de métricas visuales para los subplots de salida
    img_masked = np.zeros_like(img_enhanced)
    roi_img_masked = (roi_bilateral * best_mask_otsu_global[y_start:y_end, x_start:x_end]).astype(np.uint8)
    img_masked[y_start:y_end, x_start:x_end] = roi_img_masked

    img_bilateral = img_enhanced.copy()
    img_bilateral[y_start:y_end, x_start:x_end] = roi_bilateral

    return (
        best_mask_otsu_global,
        best_mask_rg_global,
        best_met_otsu,
        best_met_rg,
        img_bilateral,
        img_masked
    )

# Extracción de características

def extraer_caracteristicas(image_gray, binary_mask):
    from scipy.stats import skew, kurtosis

    # Formato binario estricto de 0 y 255
    mask_u8 = (binary_mask > 0).astype(np.uint8) * 255

    pixel_vals = image_gray[mask_u8 > 0].astype(float)
    if len(pixel_vals) == 0:
        return None

    #Contraste de interfaz/ borde de transición
    kernel = np.ones((3, 3), np.uint8)
    dilatada = cv2.dilate(mask_u8, kernel)
    erosionada = cv2.erode(mask_u8, kernel)

    # Capa interna: Píxeles periféricos del tumor que limitan con el exterior
    capa_interna_tumor = (mask_u8 == 255) & (erosionada == 0)

    # Capa externa: Píxeles sanos que rodean inmediatamente al tumor
    capa_externa_sana = (dilatada == 255) & (mask_u8 == 0)

    if np.any(capa_interna_tumor) and np.any(capa_externa_sana):
        intensidades_internas = image_gray[capa_interna_tumor].astype(float)
        intensidades_externas = image_gray[capa_externa_sana].astype(float)

        media_interna = np.mean(intensidades_internas)
        media_externa = np.mean(intensidades_externas)

        # Contraste local como la diferencia absoluta de nivel de gris en la frontera
        contraste_interfaz = abs(media_interna - media_externa)
    else:
        contraste_interfaz = 0.0

    labeled = label(mask_u8)
    props   = regionprops(labeled)
    if len(props) == 0:
        return None
    region = max(props, key=lambda r: r.area)

    ys, xs = np.where(mask_u8 > 0)
    y0, y1 = ys.min(), ys.max() + 1
    x0, x1 = xs.min(), xs.max() + 1
    roi    = image_gray[y0:y1, x0:x1]
    roi_q  = ((roi * (mask_u8[y0:y1, x0:x1] // 255)) // 4).astype(np.uint8)

    glcm   = graycomatrix(roi_q, distances=[1,2],
                          angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
                          levels=64, symmetric=True, normed=True)
    perim = region.perimeter
    area  = region.area

    return dict(
        area=area, perimetro=perim,
        circularidad=(4*np.pi*area)/(perim**2+1e-8),
        excentricidad=region.eccentricity,
        solidez=region.solidity, extent=region.extent,
        int_mean=np.mean(pixel_vals), int_std=np.std(pixel_vals),
        int_min=np.min(pixel_vals),   int_max=np.max(pixel_vals),
        int_skew=skew(pixel_vals),    int_kurt=kurtosis(pixel_vals),
        contraste=graycoprops(glcm,'contrast').mean(),
        homogeneidad=graycoprops(glcm,'homogeneity').mean(),
        energia=graycoprops(glcm,'energy').mean(),
        correlacion=graycoprops(glcm,'correlation').mean(),
        disimilitud=graycoprops(glcm,'dissimilarity').mean(),
        asm=graycoprops(glcm,'ASM').mean(),
        contraste_interfaz=contraste_interfaz
    )


# Búsqueda principal de segmentaciones buenas y malas

TARGET_BUENOS  = 5
TARGET_MALOS   = 2
casos_buenos   = []
casos_malos    = []
todos_los_dice = []
todos_los_dice_otsu = []
procesadas     = 0
intentos_max   = 500

random.shuffle(image_files)
candidatos = list(image_files)
idx = 0

print("\n" + "="*65)
print("  BÚSQUEDA DE CASOS")
print(f"  Objetivo: {TARGET_BUENOS} con Dice ≥ 0.7  |  {TARGET_MALOS} con Dice < 0.7")
print("="*65)

while (len(casos_buenos) < TARGET_BUENOS or len(casos_malos) < TARGET_MALOS) \
      and procesadas < intentos_max:

    if idx >= len(candidatos):
        random.shuffle(candidatos)
        idx = 0

    img_path  = candidatos[idx]; idx += 1
    base, ext = os.path.splitext(img_path)
    mask_path = f"{base}_mask{ext}"

    if not os.path.exists(mask_path):
        continue

    raw  = cv2.imread(img_path,  cv2.IMREAD_GRAYSCALE)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if raw is None or mask is None or np.max(mask) == 0:
        continue

    filtered = cv2.medianBlur(raw, 3)
    clahe    = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    enhanced = clahe.apply(filtered)

    procesadas += 1
    mo, mr, met_o, met_rg, img_bil, img_msk = segmentar(enhanced, mask)
    if mr is None:
        continue

    dice_final = met_rg['dice']
    dice_otsu_actual = met_o['dice']

    todos_los_dice.append(dice_final)
    todos_los_dice_otsu.append(dice_otsu_actual)

    if dice_final >= 0.7 and len(casos_buenos) < TARGET_BUENOS:
        casos_buenos.append(dict(
            img_path=img_path, raw=raw, enhanced=enhanced,
            img_bilateral=img_bil, img_masked=img_msk,
            mask=mask, mask_otsu=mo, mask_rg=mr,
            met_otsu=met_o, met_rg=met_rg))
        print(f"  [BUENO  {len(casos_buenos)}/{TARGET_BUENOS}] "
              f"Dice={dice_final:.4f}  (Otsu={met_o['dice']:.3f})"
              f"  →  {img_path.split('/')[-1]}")

    elif dice_final < 0.7 and len(casos_malos) < TARGET_MALOS:
        casos_malos.append(dict(
            img_path=img_path, raw=raw, enhanced=enhanced,
            img_bilateral=img_bil, img_masked=img_msk,
            mask=mask, mask_otsu=mo, mask_rg=mr,
            met_otsu=met_o, met_rg=met_rg))
        print(f"  [MALO   {len(casos_malos)}/{TARGET_MALOS}] "
              f"Dice={dice_final:.4f}  (Otsu={met_o['dice']:.3f})"
              f"  →  {img_path.split('/')[-1]}")

print(f"\n  Imágenes procesadas : {procesadas}")
print(f"  Buenos encontrados  : {len(casos_buenos)}/{TARGET_BUENOS}")
print(f"  Malos encontrados   : {len(casos_malos)}/{TARGET_MALOS}")

# Plot de distribución de Dice scores

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
n_total  = len(todos_los_dice)

# Histograma region growing
ax1.hist(todos_los_dice, bins=20, color='skyblue', edgecolor='black')
ax1.axvline(0.7, color='red', linestyle='--', linewidth=2, label='Umbral Dice = 0.7')
ax1.set_xlabel("Dice Score")
ax1.set_ylabel("Cantidad de imágenes")
ax1.set_title(f"Distribución de Dice Scores (Region Growing Adaptativo)\n({n_total} imágenes analizadas)")
ax1.grid(alpha=0.3)
ax1.legend()
n_buenos_rg = np.sum(np.array(todos_los_dice) >= 0.7)
ax1.text(0.5, -0.18, f"Dice ≥ 0.7: {n_buenos_rg}/{n_total} ({100*n_buenos_rg/n_total:.1f}%)",
          ha='center', fontsize=11, fontweight='bold', transform=ax1.transAxes)

# Histograma multi-otsu
ax2.hist(todos_los_dice_otsu, bins=20, color='salmon', edgecolor='black')
ax2.axvline(0.7, color='red', linestyle='--', linewidth=2, label='Umbral Dice = 0.7')
ax2.set_xlabel("Dice Score")
ax2.set_ylabel("Cantidad de imágenes")
ax2.set_title(f"Distribución de Dice Scores (Multi-Otsu Adaptativo)\n({n_total} imágenes analizadas)")
ax2.grid(alpha=0.3)
ax2.legend()
n_buenos_otsu = np.sum(np.array(todos_los_dice_otsu) >= 0.7)
ax2.text(0.5, -0.18, f"Dice ≥ 0.7: {n_buenos_otsu}/{n_total} ({100*n_buenos_otsu/n_total:.1f}%)",
          ha='center', fontsize=11, fontweight='bold', transform=ax2.transAxes)

plt.tight_layout()
plt.savefig("distribucion_dice_scores.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"\n[INFO] Histograma doble guardado como 'distribucion_dice_scores.png'")


# Extracción de características

print("\n" + "="*65)
print("  EXTRACCIÓN DE CARACTERÍSTICAS — MÁSCARA FINAL (Region Growing)")
print("="*65)

todos_los_casos = casos_buenos + casos_malos
etiquetas       = ["BUENO"] * len(casos_buenos) + ["MALO"] * len(casos_malos)

for i, (caso, etiq) in enumerate(zip(todos_los_casos, etiquetas)):
    feats = extraer_caracteristicas(caso['enhanced'], caso['mask_rg'])
    caso['feats'] = feats
    if feats:
        print(f"\n  [{etiq}] Caso {i+1} — Dice={caso['met_rg']['dice']:.4f}")
        print(f"    Área: {feats['area']} px²  |  Circularidad: {feats['circularidad']:.4f}"
              f"  |  Solidez: {feats['solidez']:.4f}")
        print(f"    Intensidad media: {feats['int_mean']:.2f}  ±  {feats['int_std']:.2f}")
        print(f"    [INTERFAZ] Contraste de Borde Transicional: {feats['contraste_interfaz']:.4f} u.g.")
        print(f"    GLCM — Contraste: {feats['contraste']:.4f}"
              f"  |  Homogeneidad: {feats['homogeneidad']:.4f}"
              f"  |  Energía: {feats['energia']:.4f}")

# Visualización

print("\n[INFO] Generando figura de resultados...")

n_casos = len(todos_los_casos)
fig, axes = plt.subplots(n_casos, 6, figsize=(24, 4 * n_casos))
fig.patch.set_facecolor('#0d0d0d')
if n_casos == 1:
    axes = axes[np.newaxis, :]

for row, (caso, etiq) in enumerate(zip(todos_los_casos, etiquetas)):
    mask_rg  = caso['mask_rg']
    enhanced = caso['enhanced']
    gt_bool  = (caso['mask'] > 0)
    met      = caso['met_rg']
    color_t  = '#00d26a' if etiq == "BUENO" else '#ff4444'

    axes[row, 0].imshow(enhanced, cmap='gray')
    axes[row, 0].set_title(f"[{etiq}] CLAHE", color=color_t, fontsize=8)
    axes[row, 0].axis('off')

    axes[row, 1].imshow(caso['img_bilateral'], cmap='gray')
    axes[row, 1].set_title("Filtro Bilateral", color='white', fontsize=8)
    axes[row, 1].axis('off')

    axes[row, 2].imshow(caso['mask_otsu'], cmap='hot')
    axes[row, 2].set_title(f"Multi-Otsu\nDice={caso['met_otsu']['dice']:.3f}",
                            color='white', fontsize=8)
    axes[row, 2].axis('off')

    axes[row, 3].imshow(caso['img_masked'], cmap='gray')
    axes[row, 3].set_title("Bilateral × Otsu", color='white', fontsize=8)
    axes[row, 3].axis('off')

    axes[row, 4].imshow(enhanced, cmap='gray')
    axes[row, 4].imshow(np.ma.masked_where(mask_rg == 0, mask_rg),
                        cmap='Reds', alpha=0.55)
    axes[row, 4].set_title(f"Region Growing\nDice={met['dice']:.3f}",
                            color='white', fontsize=8)
    axes[row, 4].axis('off')

    comp   = np.zeros((*enhanced.shape, 3), dtype=np.uint8)
    pred_b = mask_rg.astype(bool)
    comp[pred_b  &  gt_bool] = [0,   200,  80]
    comp[pred_b  & ~gt_bool] = [220,  50,  50]
    comp[~pred_b &  gt_bool] = [50,  100, 220]
    axes[row, 5].imshow(comp)
    axes[row, 5].set_title(f"GT vs Pred\nIoU={met['iou']:.3f}", color='white', fontsize=8)
    axes[row, 5].axis('off')

    if row == 0:
        patches = [
            mpatches.Patch(color=(0/255,200/255,80/255),   label='TP'),
            mpatches.Patch(color=(220/255,50/255,50/255),  label='FP'),
            mpatches.Patch(color=(50/255,100/255,220/255), label='FN'),
        ]
        axes[row, 5].legend(handles=patches, loc='lower right', fontsize=7,
                             facecolor='#222', labelcolor='white', framealpha=0.8)

    for a in axes[row]:
        for spine in a.spines.values():
            spine.set_edgecolor('#444')

plt.suptitle(
    "PIPELINE: ROI Recortada → CLAHE → Bilateral → Multi-Otsu Local → Region Growing"
    "  |  Verde=Dice≥0.7  /  Rojo=Dice<0.7",
    color='white', fontsize=10, fontweight='bold', y=1.005)
plt.tight_layout()
plt.savefig("resultados_segmentacion.png", dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
plt.show()
print("\n[INFO] Figura guardada como 'resultados_segmentacion.png'")

# Tabla de resumen final

print("\n" + "="*75)
print(f"  {'#':<3} {'Etiq.':<7} {'Archivo':<35} {'Dice':>6} {'IoU':>6}")
print("-"*75)
for i, (caso, etiq) in enumerate(zip(todos_los_casos, etiquetas)):
    m     = caso['met_rg']
    fname = caso['img_path'].split('/')[-1][:34]
    print(f"  {i+1:<3} {etiq:<7} {fname:<35} {m['dice']:>6.4f} {m['iou']:>6.4f}")
print("="*75)
print("\n[INFO] Pipeline finalizado correctamente.")